In [ ]:
from dotenv import load_dotenv

load_dotenv() 

import os
print("Tracing enabled:", os.getenv("LANGSMITH_TRACING"))
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage, ToolMessage
from pydantic import BaseModel
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, wrap_tool_call, ToolCallRequest, dynamic_prompt
#Handlet tool error handling




@wrap_tool_call
def t1(req: ToolCallRequest, handler)-> ToolMessage:

    try: 
        print("in try block")
        handler(req)
        res = ToolMessage(content=f"The tool message generated by Akhil as Dummy, The city weather curently is Awesome", tool_call_id = req.runtime.tool_call_id)
        print(res)       
        return res
    except Exception as e:
        return ToolMessage(content=f"The tool generated the Error {e}, so please check youtr input", tool_call_id = req.runtime.tool_call_id)



@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

#model
basic_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

advanced_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3.5:397b-cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

# #Choose Model dynamically
# @wrap_model_call
# def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
#     print("dynamic_model_selection middleware invoked!")
#     msg:str = request.messages[0].content
#     dynamic_model = request.model
#     if  msg.find("simple")< 0:
#          print("in if")
#          dynamic_model = advanced_model
#     else: 
#         print("in else")
#         dynamic_model = request.model

#     res = handler(request.override(model=dynamic_model))    
#     print("dynamic_model_selection middleware Done!")
#     print(res)
#     return res

# @wrap_model_call
# def dynamic_tool_selection(request: ModelRequest, handler) -> ModelResponse:    
#     print("dynamic_tool_selection middleware invoked!")
#     tool_list = request.tools
#     msg:str = request.messages[0].content
#     dynamic_model = request.model
#     if  msg.find("weather")< 0:
#          print("in if")
#          tool_list = []
#     else: 
#         print("in else")
#         tool_list = [get_weather_city]

#     res = handler(request.override(tools=tool_list))
#     print("dynamic_tool_selection middleware Done!")
#     print(res)
#     return res


tool_list = [get_weather_city]


@dynamic_prompt
def d1(req: ModelRequest) -> str:
    if (req.messages[0].content.find("test") > 0):
        return "Act as a Software tester & provide your answer without using techinal jargons"
    else: 
        return req.system_message.content



@wrap_model_call
def m1(req:ModelRequest, handler)-> ModelResponse:
    print(req.system_message.content)
    return handler(req)


agent = create_agent(
    name= "PodTest Agent",
    system_prompt = "Act as a Developer & provide your answer in techinal jargons",
    model=basic_model,
    tools=tool_list,
    middleware=[ d1, m1]   
)





prompt = PromptTemplate.from_template("What is Software test case?")
promptValue = prompt.invoke({})

respo = agent.invoke({"messages": [str(promptValue)]})
print(respo)

In [ ]:
from dotenv import load_dotenv

load_dotenv() 

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.agents.middleware import AgentState
from langgraph.types import Command


from langchain_community.tools import DuckDuckGoSearchRun

model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + ''}
        }
    ) 

class CustomState(AgentState):
    countryName: str
    countryWeather: str
    countryCapital: str


# ── Helper: distil raw search text → short answer via LLM ────────────────────
def extract_short(question: str, raw: str) -> str:
    prompt = (
        f"Answer this question in as few words as possible "
        f"(ideally one word or short phrase). "
        f"Return ONLY the answer, nothing else.\n\n"
        f"Question: {question}\nText:\n{raw}"
    )
    return model.invoke([HumanMessage(content=prompt)]).content.strip()

@tool
def get_weather_country(country: str, runtime: ToolRuntime) -> Command:
    """Get the current weather for a country
    
    Args:
        country: The name of the country for which to get weather information
    
    Returns:
        A string describing the weather in the specified country
    """

    search = DuckDuckGoSearchRun()
    raw = search.invoke(f"current weather in {country} today")    
    short = extract_short(f"What is the current weather in {country}?", raw)

    tool_message = ToolMessage(
        content=f"Weather in {country}: {short}",
        tool_call_id=runtime.tool_call_id  # Links back to the specific tool call
    )

    return Command(update={
        "messages": [tool_message],
        "countryName":country,
        "countryWeather": short})    

@tool
def get_country_capital(country: str, runtime: ToolRuntime) -> Command:
    """Get the capital for a country
    
    Args:
        country: The name of the country for which to get the capital
    
    Returns:
        A string mentioning the capital Name
    """

    search = DuckDuckGoSearchRun()
    raw = search.invoke(f"capital city of {country}")
    
    short = extract_short(f"What is the capital city of {country}?", raw)

    tool_message = ToolMessage(
        content=f"Capital of {country}: {short}",
        tool_call_id=runtime.tool_call_id
    )


    runtime.state.countryName

    return Command(update={
        "messages": [tool_message],
        "countryCapital": short
        
        })    


tool_list= [get_weather_country, get_country_capital]

sysMsg = SystemMessage(content="""
    Act as a weather specialist, User will ask for details about a country.
    ## Workflow (always follow this order):
    1. Call `get_weather_country` with the country Name
    2. Call `get_country_capital` with the country Name.    
""")

#3. Summarise both results concisely.   

agent = create_agent(
    name= "PodTest Weather and Capital Agent",  
    system_prompt=sysMsg , 
    model=model,
    tools=tool_list,
    state_schema=CustomState
)

res = agent.invoke(
    {
        "messages": [HumanMessage(content="Give me details about country India")],
        'countryName': "India"
    }
)

print(res)

In [ ]:
from dotenv import load_dotenv

load_dotenv() 

import os
print("Tracing enabled:", os.getenv("LANGCHAIN_TRACING_V2"))
print("Tracing enabled:", os.getenv("LANGSMITH_ENDPOINT"))
print("Tracing enabled:", os.getenv("LANGSMITH_API_KEY"))
print("Tracing enabled:", os.getenv("LANGSMITH_PROJECT"))

from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        #"headers": {'Authorization': 'Bearer ' + str(os.getenv("OLLAMA_API_KEY"))}
        "headers": {'Authorization': 'Bearer ' + ""}
        
        }
    )


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]})